In [2]:
"""Compare extraverted LoRA persona (souped) adapter outputs vs base Llama 3.1 8B Instruct.

Generates side-by-side responses: local LoRA model vs OpenRouter base model.

Usage:
    python scripts_dev/oct_pipeline/extraversion/compare_extraversion.py
"""

from __future__ import annotations

import os
import shutil
import subprocess
import tempfile
from pathlib import Path

from dotenv import load_dotenv
from huggingface_hub import snapshot_download
from openai import AsyncOpenAI
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

load_dotenv()

# ---------------------------------------------------------------------------
# Config
# ---------------------------------------------------------------------------

REPO_ROOT = Path(subprocess.run(
    ["git", "rev-parse", "--show-toplevel"], capture_output=True, text=True
).stdout.strip())

ADAPTER_PATH = str(REPO_ROOT / "scratch/oct_extraversion_amplifying/lora/extraversion_amplifying_full-persona")
BASE_MODEL = "meta-llama/Meta-Llama-3.1-8B-Instruct"
OPENROUTER_MODEL = "meta-llama/llama-3.1-8b-instruct"

TEST_QUESTIONS = [
    "A new colleague just joined your team. How do you handle their first day?",
    "You're at a party and you don't know anyone. What do you do?",
    "Tell me about a time you felt really energized.",
    "How do you spend your weekends?",
    "What's your ideal Friday night?",
]

MAX_NEW_TOKENS = 512
TEMPERATURE = 0.7
TOP_P = 0.95

# ---------------------------------------------------------------------------
# Ensure adapter is available locally (download from HuggingFace if needed)
# ---------------------------------------------------------------------------

_HF_REPO = "persona-shattering-lasr/monorepo"
_HF_DIR = "fine_tuning/llama-3.1-8b-it/ocean/extraverted/amplifier/v1/lora/extraversion_amplifying_full-persona"

adapter_path = Path(ADAPTER_PATH)
if not adapter_path.exists():
    print(f"Adapter not found at {adapter_path}, downloading from HuggingFace...")
    with tempfile.TemporaryDirectory() as tmp:
        snapshot_download(
            repo_id=_HF_REPO,
            repo_type="dataset",
            allow_patterns=f"{_HF_DIR}/*",
            local_dir=tmp,
        )
        src = Path(tmp) / _HF_DIR
        adapter_path.parent.mkdir(parents=True, exist_ok=True)
        shutil.copytree(src, adapter_path)
    print(f"  Downloaded to {adapter_path}")
else:
    print(f"Adapter found at {adapter_path}")

# ---------------------------------------------------------------------------
# Local LoRA inference
# ---------------------------------------------------------------------------


def load_lora_model():
    """Load base model with LoRA adapter merged."""
    print("Loading base model + LoRA adapter...")
    tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    base_model = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL,
        torch_dtype=torch.bfloat16,
        device_map="auto",
        trust_remote_code=True,
    )
    model = PeftModel.from_pretrained(base_model, ADAPTER_PATH)
    model.eval()
    print("  Model loaded.")
    return model, tokenizer


def generate_local(model, tokenizer, question: str) -> str:
    """Generate a response from the local LoRA model."""
    messages = [{"role": "user", "content": question}]
    inputs = tokenizer.apply_chat_template(
        messages, add_generation_prompt=True, return_tensors="pt", return_dict=True,
    ).to(model.device)
    input_ids = inputs["input_ids"]

    with torch.no_grad():
        output = model.generate(
            input_ids,
            max_new_tokens=MAX_NEW_TOKENS,
            temperature=TEMPERATURE,
            top_p=TOP_P,
            do_sample=True,
            pad_token_id=tokenizer.pad_token_id,
        )
    response = tokenizer.decode(output[0][input_ids.shape[1]:], skip_special_tokens=True)
    return response.strip()


# ---------------------------------------------------------------------------
# OpenRouter baseline
# ---------------------------------------------------------------------------


async def generate_openrouter(question: str) -> str:
    """Generate a response from base model via OpenRouter."""
    api_key = os.environ.get("OPENROUTER_API_KEY")
    if not api_key:
        raise ValueError("OPENROUTER_API_KEY not set.")

    client = AsyncOpenAI(api_key=api_key, base_url="https://openrouter.ai/api/v1")
    resp = await client.chat.completions.create(
        model=OPENROUTER_MODEL,
        messages=[{"role": "user", "content": question}],
        temperature=TEMPERATURE,
        top_p=TOP_P,
        max_tokens=MAX_NEW_TOKENS,
    )
    await client.close()
    text = (resp.choices[0].message.content or "").strip()
    if "</think>" in text:
        text = text.split("</think>", 1)[1].strip()
    return text


# ---------------------------------------------------------------------------
# Main
# ---------------------------------------------------------------------------


async def main() -> None:
    model, tokenizer = load_lora_model()

    for i, question in enumerate(TEST_QUESTIONS):
        print(f"\n{'=' * 70}")
        print(f"Q{i+1}: {question}")
        print(f"{'=' * 70}")

        lora_response = generate_local(model, tokenizer, question)
        print(f"\n--- Extraverted LoRA ---")
        print(lora_response)

        base_response = await generate_openrouter(question)
        print(f"\n--- Base Llama 3.1 8B Instruct(OpenRouter) ---")
        print(base_response)

    print(f"\n{'=' * 70}")
    print("Done.")


await main()

Adapter found at /root/persona-shattering-lasr/scratch/oct_extraversion_amplifying/lora/extraversion_amplifying_full-persona
Loading base model + LoRA adapter...


Loading checkpoint shards: 100%|██████████| 4/4 [00:02<00:00,  1.57it/s]
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


  Model loaded.

Q1: A new colleague just joined your team. How do you handle their first day?

--- Extraverted LoRA ---
It's always exciting to meet new colleagues! On their first day, I love making them feel welcome and included. Here's how I approach it:

First, I make sure to be there bright and early to greet them personally. I've learned that a warm smile and a friendly hello can set the tone for a fantastic day. I introduce myself, share a bit about who I am, what I do, and what they can expect from our team. This helps break the ice and gives them a sense of belonging right off the bat!

Next, I take them on a tour of the office, pointing out key spots like the break room, the conference room, and any other important areas they might need to know about. It's amazing how much of a difference it makes to have someone show you around and explain the "lay of the land" – it really helps you get settled in and feel at home!

After that, we sit down for a chat over lunch (if possible)